In [1]:
import cupy as np
# import numpy as np

def rsvd(A, k, p=0, q=0):
    m, n = A.shape
    Omega = np.random.randn(n, k + p)
    Y = A @ Omega
    for _ in range(q):
        Y = A @ (A.T @ Y)
    Q, _ = np.linalg.qr(Y)
    B = Q.T @ A
    U_tilde, S, Vt = np.linalg.svd(B, full_matrices=False)
    U = Q @ U_tilde
    return U[:, :k], S[:k], Vt[:k, :]

def ten2mat(tensor, mode):
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order = 'F')

def update_cg(var, r, q, Aq, rold):
    alpha = rold / np.inner(q, Aq)
    var = var + alpha * q
    r = r - alpha * Aq
    rnew = np.inner(r, r)
    q = r + (rnew / rold) * q
    return var, r, q, rnew

def fold_G(G, Rw, Ru, Rv):
    Gt = np.zeros((Rw, Ru, Rv))
    for r in range(Rv):
        Gt[:, :, r] = G[:, r * Ru : (r + 1) * Ru]
    return Gt

In [2]:
# import numpy as np

def compute_mape(var, var_hat):
    return np.sum(np.abs(var - var_hat) / var) / var.shape[0]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])

def forecast(series, mu, W, G, U, V, horizon):
    N, t = series.shape
    d, Rv = V.shape
    Rw = W.shape[1]
    Ru = U.shape[1]
    est = np.zeros((N, horizon))
    for h in range(horizon):
        Xt = np.zeros((N, d))
        for k in range(d):
            Xt[:, k] = series[:, t - horizon + h - (k + 1)]
        est[:, h] = mu + np.einsum('nw, w -> n', W,
                                   np.einsum('wuv, uv -> w',
                                             fold_G(G, Rw, Ru, Rv), U.T @ Xt @ V))
    return est

In [3]:
def solve_mu(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, nu, dv -> uvt', X, U, V)
    C = Y[:, d :] - W @ np.einsum('wuv, uvt -> wt', fold_G(G, Rw, Ru, Rv), mat)
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = C - mu[:, np.newaxis]
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################
        mu = np.einsum('nt, nt -> n', theta, C) / np.sum(theta, axis = 1)
    return mu

In [4]:
def ell_W(temp, W):
    return np.einsum('nwk, nw -> nk', temp, W).reshape(-1, order = 'F')

def solve_W(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    var0 = np.einsum('nu, ndt, dv -> uvt', U, X, V)
    var1 = np.einsum('wuv, uvt -> wt', fold_G(G, Rw, Ru, Rv), var0)
    temp1 = np.einsum('wt, kt -> wkt', var1, var1)
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - np.einsum('nw, wt -> nt', W, var1)
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        w = np.reshape(W, -1, order = 'F')
        response = np.einsum('nt, wt -> nw', theta * data, var1).reshape(-1, order = 'F')
        temp = np.einsum('nt, wkt -> nwk', theta, temp1)
        r = response - ell_W(temp, W)
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (N, Rw), order = 'F')
            Aq = ell_W(temp, Q)
            w, r, q, rold = update_cg(w, r, q, Aq, rold)
        W = np.reshape(w, (N, Rw), order = 'F')
    return W

In [5]:
def ell_U(var0, temp1, U):
    return np.einsum('njut, jt -> nu', var0, np.einsum('jwt, wt -> jt', temp1, np.einsum('nwut, nu -> wt', var0, U)))

def solve_U(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, dv -> nvt', X, V)
    var0 = np.einsum('nvt, wuv -> nwut', mat, fold_G(G, Rw, Ru, Rv))
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - np.einsum('nw, wt -> nt', W, np.einsum('nwut, nu -> wt', var0, U))
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        temp1 = np.einsum('ij, it, ik -> jkt', W, theta, W)

        ### Conjugate Gradient ###
        u = np.reshape(U, -1, order = 'F')
        response = np.einsum('nwut, wt -> nu', var0, W.T @ (theta * data))
        r = np.reshape(response - ell_U(var0, temp1, U), -1, order = 'F')
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (N, Ru), order = 'F')
            Aq = np.reshape(ell_U(var0, temp1, Q), -1, order = 'F')
            u, r, q, rold = update_cg(u, r, q, Aq, rold)
        U = np.reshape(u, (N, Ru), order = 'F')
    return U

In [6]:
def ell_V(var0, temp1, V):
    return np.einsum('djvt, jt -> dv', var0, np.einsum('jwt, wt -> jt', temp1, np.einsum('dwvt, dv -> wt', var0, V)))

def solve_V(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, nu -> dut', X, U)
    var0 = np.einsum('dut, wuv -> dwvt', mat, fold_G(G, Rw, Ru, Rv))
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - np.einsum('nw, wt -> nt', W, np.einsum('dwvt, dv -> wt', var0, V))
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        temp1 = np.einsum('ij, it, ik -> jkt', W, theta, W)

        ### Conjugate Gradient ###
        v = np.reshape(V, -1, order = 'F')
        response = np.einsum('dwvt, wt -> dv', var0, W.T @ (theta * data))
        r = np.reshape(response - ell_V(var0, temp1, V), -1, order = 'F')
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (d, Rv), order = 'F')
            Aq = np.reshape(ell_V(var0, temp1, Q), -1, order = 'F')
            v, r, q, rold = update_cg(v, r, q, Aq, rold)
        V = np.reshape(v, (d, Rv), order = 'F')
    return V

In [7]:
def left_svd_gram(X, k):
    G = X @ X.T
    evals, evecs = np.linalg.eigh(G)
    idx = np.argsort(evals)[::-1][:k]
    return evecs[:, idx], np.sqrt(np.clip(evals[idx], 0, None))

In [8]:
import cupy as np
import time

def huber_tar(Y, p, delta, d, Rw, Ru, Rv, max_iter = 50):
    N, T = Y.shape
    ##############
    mu = np.zeros(N)
    factor, _, _ = rsvd(Y[:, d :], Rw)
    W = factor[:, : Rw]
    X = np.zeros((N, d * (T - d)))
    for k in range(d):
        X[:, k * (T - d) : (k + 1) * (T - d)] = Y[:, d - k - 1 : T - k - 1]
    factor, _, _ = rsvd(X, Ru)
    U = factor[:, : Ru]
    X = np.zeros((d, N * (T - d)))
    for k in range(d):
        X[k, :] = Y[:, d - k - 1 : T - k - 1].reshape(-1, order = 'F')
    if N * (T - d) > 1e+7:
        V, _ = left_svd_gram(X, Rv)
    else:
        factor, _, _ = rsvd(X, Rv)
        V = factor[:, : Rv]
    tensor_nnb = np.zeros((Rw, Ru, Rv))
    idx = np.arange(Rw)
    tensor_nnb[idx, idx, idx] = 1
    G = ten2mat(tensor_nnb, 0)
    ###############
    for iter in range(max_iter):
        mu = solve_mu(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p)
        W = solve_W(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p)
        U = solve_U(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p)
        V = solve_V(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p)
    return mu, W, G, U, V

In [12]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 168
    d = 7 * 24
    for R in [3]:
        for delta in [40]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, G, U, V = huber_tar(np.array(data[:, : 17 * 24]), p, delta, d, R, R, R, 50)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, 24 * 17 : 24 * 24])
            est = forecast(np.array(data[:, : 24 * 24]), mu, W, G, U, V, horizon)
            pos = np.where(ground > 0)
            print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
            print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, G, U, V, horizon)
    pos = np.where(ground > 0)
    print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
    print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
    print()

p = 1
Rank = 3
Delta = 40
Running time: 54 seconds
MAPE: 2.84262
RMSE: 2240.75

MAPE: 4.98002
RMSE: 2752.05

p = 0.6666666666666666
Rank = 3
Delta = 40
Running time: 54 seconds
MAPE: 2.45282
RMSE: 1976.57

MAPE: 4.51635
RMSE: 2608.64

p = 0.5
Rank = 3
Delta = 40
Running time: 54 seconds
MAPE: 2.7491
RMSE: 2226.13

MAPE: 4.91291
RMSE: 2829.43

p = 0.3333333333333333
Rank = 3
Delta = 40
Running time: 54 seconds
MAPE: 2.79851
RMSE: 2045.74

MAPE: 5.11237
RMSE: 2710.25

p = 0
Rank = 3
Delta = 40
Running time: 54 seconds
MAPE: 2.54106
RMSE: 1708.77

MAPE: 4.53316
RMSE: 2287.45



In [13]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 168
    d = 7 * 24
    for R in [4]:
        for delta in [40]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, G, U, V = huber_tar(np.array(data[:, : 17 * 24]), p, delta, d, R, R, R, 50)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, 24 * 17 : 24 * 24])
            est = forecast(np.array(data[:, : 24 * 24]), mu, W, G, U, V, horizon)
            pos = np.where(ground > 0)
            print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
            print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, G, U, V, horizon)
    pos = np.where(ground > 0)
    print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
    print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
    print()

p = 1
Rank = 4
Delta = 40
Running time: 62 seconds
MAPE: 2.62058
RMSE: 2643.32

MAPE: 5.85785
RMSE: 4037.29

p = 0.6666666666666666
Rank = 4
Delta = 40
Running time: 62 seconds
MAPE: 2.92626
RMSE: 2754.67

MAPE: 6.33864
RMSE: 4198.75

p = 0.5
Rank = 4
Delta = 40
Running time: 62 seconds
MAPE: 2.92341
RMSE: 2720.31

MAPE: 5.98645
RMSE: 4049.39

p = 0.3333333333333333
Rank = 4
Delta = 40
Running time: 61 seconds
MAPE: 2.9397
RMSE: 2629.72

MAPE: 5.68724
RMSE: 3838.85

p = 0
Rank = 4
Delta = 40
Running time: 62 seconds
MAPE: 3.0427
RMSE: 2456.46

MAPE: 5.3867
RMSE: 3578.07



In [9]:
import numpy as np

data = np.load('daymet_tmax_na_2010.npz')['arr_0']
for t in [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]:
    data = np.append(data, np.load('daymet_tmax_na_20{}.npz'.format(t))['arr_0'], axis = 1)

In [10]:
import time
import cupy as np

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 365
    d = 30
    for R in [5]:
        for delta in [1]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, G, U, V = huber_tar(np.array(data[:, : - (366 + 365)]), p, delta, d, R, R, R, 50)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, - (366 + 365) : - 365])
            est = forecast(np.array(data[:, : - 365]), mu, W, G, U, V, horizon)
            pos = np.where(ground != 0)
            print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
            print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, G, U, V, horizon)
    pos = np.where(ground != 0)
    print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
    print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
    print()

p = 1
Rank = 5
Delta = 1
Running time: 511 seconds
MAPE: 0.221494
RMSE: 4.97108

MAPE: 0.504098
RMSE: 4.38119

p = 0.6666666666666666
Rank = 5
Delta = 1
Running time: 504 seconds
MAPE: 0.216025
RMSE: 4.78503

MAPE: 0.722258
RMSE: 4.27086

p = 0.5
Rank = 5
Delta = 1
Running time: 504 seconds
MAPE: 0.223735
RMSE: 4.89176

MAPE: 0.689343
RMSE: 4.42506

p = 0.3333333333333333
Rank = 5
Delta = 1
Running time: 504 seconds
MAPE: 0.225317
RMSE: 5.0249

MAPE: 0.544867
RMSE: 4.42499

p = 0
Rank = 5
Delta = 1
Running time: 503 seconds
MAPE: 0.210223
RMSE: 4.5207

MAPE: 0.598311
RMSE: 3.95492



In [11]:
import time
import cupy as np

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 365
    d = 30
    for R in [6]:
        for delta in [1]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, G, U, V = huber_tar(np.array(data[:, : - (366 + 365)]), p, delta, d, R, R, R, 50)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, - (366 + 365) : - 365])
            est = forecast(np.array(data[:, : - 365]), mu, W, G, U, V, horizon)
            pos = np.where(ground != 0)
            print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
            print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, G, U, V, horizon)
    pos = np.where(ground != 0)
    print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
    print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
    print()

p = 1
Rank = 6
Delta = 1
Running time: 737 seconds
MAPE: 0.232738
RMSE: 5.12987

MAPE: 0.330215
RMSE: 4.62703

p = 0.6666666666666666
Rank = 6
Delta = 1
Running time: 736 seconds
MAPE: 0.228168
RMSE: 4.9713

MAPE: 0.305394
RMSE: 4.41493

p = 0.5
Rank = 6
Delta = 1
Running time: 736 seconds
MAPE: 0.229436
RMSE: 5.00261

MAPE: 0.303638
RMSE: 4.34627

p = 0.3333333333333333
Rank = 6
Delta = 1
Running time: 736 seconds
MAPE: 0.233119
RMSE: 5.0389

MAPE: 0.370005
RMSE: 4.39285

p = 0
Rank = 6
Delta = 1
Running time: 736 seconds
MAPE: 0.195619
RMSE: 4.29197

MAPE: 0.289763
RMSE: 3.70254

